In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# NOMIS Data
# https://www.nomisweb.co.uk/query/construct/submit.asp?menuopt=201&subcomp=

path = Path("~/Data/nomis.tsv")
data = pd.read_csv(path, sep="\t", encoding="utf-8")[
    ["Projected Yea", "Age", "value"]
].rename(
    columns={"Projected Yea": "year", "Age": "age_group", "value": "value"}
)
# pivot but do not set age as index yet
data = data.pivot(
    index="age_group", columns="year", values="value"
).reset_index()

# map to age groups from nts: ['70+', '50-70', '20-30', '30-40', '40-50', '11-16', '16-20', '5-11', '<5']
age_range_dict = {
    "Age 0 - 4": "<5",
    "Age 5-9": "5-11",
    "Age 10-14": "11-16",
    "Age 15-19": "16-20",
    "Age 20-24": "20-30",
    "Age 25-29": "20-30",
    "Age 30-34": "30-40",
    "Age 35-39": "30-40",
    "Age 40-44": "40-50",
    "Age 45-49": "40-50",
    "Age 50-54": "50-70",
    "Age 55-59": "50-70",
    "Age 60-64": "50-70",
    "Age 65-69": "50-70",
    "Age 70-74": "70+",
    "Age 75-79": "70+",
    "Age 80-84": "70+",
    "Age 85-89": "70+",
    "Age 90-94": "70+",
    "Age 95-99": "70+",
    "Age 100-104": "70+",
    "Age 105+": "70+",
}

data.age_group = data.age_group.map(age_range_dict)
data = data.groupby("age_group").sum()  # ensure all ages are present

data

year,2023,2030,2040,2050
age_group,,,,
11-16,4001402,3804787,3511260,3584389
16-20,3908458,4133760,3669519,3674434
20-30,8462438,9103127,9166601,8408349
30-40,9160077,9777469,10050284,10117607
40-50,8265174,9282996,10074245,10354985
5-11,3772100,3463684,3468318,3563675
50-70,16372643,16642546,16943900,18720119
70+,9197875,10358099,12544986,13386748
<5,3432680,3364665,3437461,3467538


In [3]:
# build nomis age attributes
write_path = Path("../tmp")
for year in data.columns:
    ages = []
    for age, n in zip(data.index, data[year]):
        ages.extend([age] * int(n / 100))  # 1% sample size
    df = pd.DataFrame(ages, index=range(len(ages)), columns=["age_group"])
    df.index.name = "pid"
    df.to_csv(write_path / f"nomis_ages_{year}.csv")

In [4]:
# fake attributes

route = Path("../tmp")
ref_path = route / "nts_attributes_2023.csv"
attributes = pd.read_csv(ref_path)
attributes.age_group.value_counts()

age_group
50-70    17731
70+      10291
40-50     7681
30-40     7551
20-30     4568
5-11      3563
11-16     3489
<5        2454
16-20     1937
Name: count, dtype: int64

In [5]:
age_mapping = {
    1: 0,
    2: 1,
    3: 3,
    4: 5,
    5: 11,
    6: 16,
    7: 17,
    8: 18,
    9: 19,
    10: 20,
    11: 21,
    12: 26,
    13: 30,
    14: 40,
    15: 50,
    16: 60,
    17: 65,
    18: 70,
    19: 75,
    20: 80,
    21: 85,
}
age_group_mapping = {
    1: "<5",
    2: "<5",
    3: "<5",
    4: "5-11",
    5: "11-16",
    6: "16-20",
    7: "16-20",
    8: "16-20",
    9: "16-20",
    10: "20-30",
    11: "20-30",
    12: "20-30",
    13: "30-40",
    14: "40-50",
    15: "50-70",
    16: "50-70",
    17: "50-70",
    18: "70+",
    19: "70+",
    20: "70+",
    21: "70+",
}
{age_mapping[k]: age_group_mapping[k] for k in age_mapping.keys()}

{0: '<5',
 1: '<5',
 3: '<5',
 5: '5-11',
 11: '11-16',
 16: '16-20',
 17: '16-20',
 18: '16-20',
 19: '16-20',
 20: '20-30',
 21: '20-30',
 26: '20-30',
 30: '30-40',
 40: '40-50',
 50: '50-70',
 60: '50-70',
 65: '50-70',
 70: '70+',
 75: '70+',
 80: '70+',
 85: '70+'}

In [6]:
# age population

plus_10_mapper = {
    10: "5-11",
    11: "11-16",
    13: "11-16",
    15: "11-16",
    21: "20-30",
    26: "20-30",
    27: "20-30",
    28: "20-30",
    29: "20-30",
    30: "20-30",
    31: "30-40",
    36: "30-40",
    40: "40-50",
    50: "50-70",
    60: "50-70",
    70: "70+",
    75: "70+",
    80: "70+",
    85: "70+",
    90: "70+",
    95: "70+",
}
plus_20_mapper = {
    20: "20-30",
    21: "20-30",
    23: "20-30",
    25: "20-30",
    31: "30-40",
    36: "30-40",
    37: "30-40",
    38: "30-40",
    39: "30-40",
    40: "40-50",
    41: "40-50",
    46: "40-50",
    50: "50-70",
    60: "50-70",
    70: "70+",
    80: "70+",
    85: "70+",
    90: "70+",
    95: "70+",
    100: "70+",
    105: "70+",
}
plus_30_mapper = {
    30: "30-40",
    31: "30-40",
    33: "30-40",
    35: "30-40",
    41: "40-50",
    46: "40-50",
    47: "40-50",
    48: "40-50",
    49: "40-50",
    50: "50-70",
    51: "50-70",
    56: "50-70",
    60: "50-70",
    70: "70+",
    80: "70+",
    90: "70+",
    95: "70+",
    100: "70+",
    105: "70+",
    110: "70+",
    115: "70+",
}
for age, mapper in [
    (10, plus_10_mapper),
    (20, plus_20_mapper),
    (30, plus_30_mapper),
]:
    aged = attributes.copy()
    aged.age = aged.age + age
    aged.age_group = aged.age.map(mapper)
    name = f"nts_attributes_2023_aged{age}.csv"
    aged.to_csv(route / name, index=False)
    print(f"Aged {age}: {aged.isna().sum().sum()} NAs => {name}")
    print(aged.age_group.value_counts())

Aged 10: 0 NAs => nts_attributes_2023_aged10.csv
age_group
70+      19381
50-70    16322
40-50     7551
20-30     5761
11-16     5665
30-40     4233
5-11       352
Name: count, dtype: int64
Aged 20: 0 NAs => nts_attributes_2023_aged20.csv
age_group
70+      28022
50-70    15232
20-30     6017
30-40     5426
40-50     4568
Name: count, dtype: int64
Aged 30: 0 NAs => nts_attributes_2023_aged30.csv
age_group
70+      35703
50-70    12119
30-40     6017
40-50     5426
Name: count, dtype: int64
